In [77]:
import argparse
import yaml
import numpy as np
import pandas as pd
import xarray as xr
from datetime import timedelta
from pathlib import Path
import subprocess
from io import StringIO
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import matplotlib.pyplot as plt
import csv
from scipy.optimize import minimize

In [78]:
def load_config(config_path="config_calib.yml"):
    try:
        # Case 1: Running as a script → __file__ exists
        base_path = Path(__file__).parent
        
    except NameError:
        # Case 2: Running in Jupyter → __file__ does NOT exist
        base_path = Path(os.getcwd())

    config_path = base_path / config_path

    if not config_path.exists():
        raise FileNotFoundError(f"Config file not found: {config_path}")

    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)

    return cfg

# Logging + Export Utilities

In [79]:
def init_logging(log_file):
    with open(log_file, "w") as f:
        writer = csv.writer(f)
        writer.writerow(["iteration", "rmse", "bias"])

def append_logging(log_file, iteration, rmse, bias):
    with open(log_file, "a") as f:
        writer = csv.writer(f)
        writer.writerow([iteration, rmse, bias])

def export_netcdf(merged_df, outfile):
    ds = xr.Dataset(
        {
            "hs": (("time",), merged_df["hs"].values),
            "swe_obs": (("time",), merged_df["swe_obs"].values),
            "swe_mod": (("time",), merged_df["swe_mod"].values),
        },
        coords={"time": merged_df["date"].values},
    )
    ds.to_netcdf(outfile)
    print(f"✓ Saved NetCDF → {outfile}")

def plot_rmse_bias(log_file, outfile):
    df = pd.read_csv(log_file)
    plt.figure(figsize=(8,5))
    plt.plot(df["iteration"], df["rmse"], label="RMSE")
    plt.plot(df["iteration"], df["bias"], label="Bias")
    plt.xlabel("Iteration")
    plt.ylabel("Value")
    plt.title("RMSE & Bias during Optimization")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outfile, dpi=200)
    print(f"✓ Saved plot → {outfile}")

# Data preparation

In [80]:
def build_fit_tibble(d_obs_fit, start_month=8):
    rows = []
    for name, df in d_obs_fit.items():
        df = df.copy()
        blocks = [(d.year - 1 if d.month < start_month else d.year) for d in df.index]

        tib = pd.DataFrame({
            "date": df.index,
            "name": name,
            "hs": df["hs"].values,
            "swe_obs": df["swe_obs"].values,
            "block": blocks,
        })
        rows.append(tib)

    return pd.concat(rows, ignore_index=True)

def split_into_seasons(station_dfs, season_start="-08-01", season_end="-07-31"):
    d_obs_fit, d_obs_val = {}, {}

    for station, df in station_dfs.items():
        df = df.copy().set_index("date")
        years = df.index.year.unique()

        winters_fit, winters_val = [], []

        for y in years:
            start = pd.to_datetime(f"{y}{season_start}")
            end = pd.to_datetime(f"{y+1}{season_end}") + timedelta(days=1)
            w = df[(df.index >= start) & (df.index < end)]

            if len(w) < 150:
                continue

            if w.index[0].year % 2 == 0:
                winters_fit.append(w)
            else:
                winters_val.append(w)

        if winters_fit:
            d_obs_fit[station] = pd.concat(winters_fit)
        if winters_val:
            d_obs_val[station] = pd.concat(winters_val)

    return d_obs_fit, d_obs_val


#temproray filers ar not saved 

# Δ-SNOW runner (parallel)

In [81]:
def run_single_block(args):
    station, block, df_sub, r_script_path, params, temp_dir = args

    df_sub = df_sub.sort_values("date")
    full_range = pd.date_range(df_sub["date"].min(), df_sub["date"].max(), freq="D")
    df_sub = df_sub.set_index("date").reindex(full_range).rename_axis("date").reset_index()
    df_sub["hs"] = df_sub["hs"].fillna(0)

    input_csv = Path(temp_dir) / f"hs_input_{station}_{block}.csv"
    df_sub[["date", "hs"]].to_csv(input_csv, index=False)

    cmd = ["Rscript", r_script_path, "--in", str(input_csv)]
    for k, v in params.items():
        cmd += [f"--{k}", str(v)]

    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print(f"ERROR {station} block {block}: {p.stderr}")
        return None

    try:
        out = pd.read_csv(StringIO(p.stdout), parse_dates=["date"])
        out["station"] = station
        out["block"] = block
        return out
    except:
        return None

def evaluate_model_parallel(df_all, r_script_path, params, workers, temp_dir="/tmp"):

    stations = df_all["name"].unique()
    blocks = df_all["block"].unique()

    jobs = []
    for st in stations:
        for blk in blocks:
            df_sub = df_all[(df_all["name"] == st) & (df_all["block"] == blk)]
            if len(df_sub) < 100:
                continue
            jobs.append((st, blk, df_sub.copy(), r_script_path, params, temp_dir))

    with Pool(workers) as pool:
        results = list(tqdm(pool.imap(run_single_block, jobs), total=len(jobs), desc="Δ-SNOW blocks"))

    results = [r for r in results if r is not None]
    if not results:
        raise RuntimeError("No Δ-SNOW output.")

    out = pd.concat(results, ignore_index=True)

    merged = df_all.merge(out.rename(columns={"station": "name"}), on=["date", "name"], how="inner")
    valid = merged.dropna(subset=["swe_obs", "swe_mod"])

    rmse = np.sqrt(np.mean((valid["swe_mod"] - valid["swe_obs"]) ** 2))
    bias = np.mean(valid["swe_mod"] - valid["swe_obs"])

    return merged, {"rmse": rmse, "bias": bias}

# Optimization

In [82]:
# Optimization

def optimize_params(df_fit, r_script_path, param_names, initial, bounds, scale, workers, log_file, maxiter):

    iteration = {"i": 0}

    def objective(x_scaled):
        iteration["i"] += 1
        params = {p: xv * s for p, xv, s in zip(param_names, x_scaled, scale)}

        merged, metrics = evaluate_model_parallel(df_fit, r_script_path, params, workers)
        rmse, bias = metrics["rmse"], metrics["bias"]

        append_logging(log_file, iteration["i"], rmse, bias)
        print(f"Iteration {iteration['i']}: RMSE={rmse:.3f}  Bias={bias:.3f}")

        return rmse

    x0_scaled = [v / s for v, s in zip(initial, scale)]
    bounds_scaled = [(lo / s, hi / s) for (lo, hi), s in zip(bounds, scale)]

    res = minimize(
        objective,
        x0_scaled,
        method="L-BFGS-B",
        bounds=bounds_scaled,
        options={"maxiter": maxiter},
    )

    final_params = {p: xv * s for p, xv, s in zip(param_names, res.x, scale)}
    return res, final_params


In [83]:
x0_scaled

NameError: name 'x0_scaled' is not defined


# Main


In [87]:
# Load config
cfg = load_config("/Users/jakobwerkgarner/code/mt_dsnow/calibration/calibration_py/L-BFGS-B/config_calib.yml")

# Extract paths
nc_file = cfg["paths"]["nc_file"]
r_script = cfg["paths"]["r_script"]
output_dir = Path(cfg["paths"]["output_dir"])
output_dir.mkdir(exist_ok=True)

# Optimization settings
workers = (
    max(cpu_count() - 1, 1)
    if cfg["optimization"]["workers"] == "auto"
    else cfg["optimization"]["workers"]
)


maxiter = cfg["optimization"]["maxiter"]




# Parameter definitions from config
param_section = cfg["parameters"]

param_names = param_section["names"]

In [90]:
# FIX: force-cast AND update variables
initial = [float(v) for v in param_section["initial"]]
scale   = [float(v) for v in param_section["scale"]]
bounds  = [[float(b[0]), float(b[1])] for b in param_section["bounds"]]

In [93]:
bounds

[[0.3, 0.6],
 [0.05, 0.2],
 [0.0, 1.0],
 [0.01, 10.0],
 [0.1, 2.0],
 [0.1, 2.0],
 [0.1, 2.0]]

In [ ]:





# FIX: force-cast AND update variables
initial = [float(v) for v in param_section["initial"]]
scale   = [float(v) for v in param_section["scale"]]
bounds  = [[float(b[0]), float(b[1])] for b in param_section["bounds"]]

log_file = output_dir / "calibration_log.csv"
init_logging(log_file)

print("\nLoading dataset...")
ds = xr.open_dataset(nc_file, engine="netcdf4").rename({"SWE": "swe_obs", "HS": "hs"})
ds_red = ds[["hs", "swe_obs"]]

station_dfs = {
    st: ds_red.sel(station=st).to_dataframe()
                .reset_index()
                .rename(columns={"time": "date"})
    for st in ds_red["station"].values
}

print("Splitting...")
d_fit, d_val = split_into_seasons(station_dfs)
df_fit = build_fit_tibble(d_fit)

print("\nStarting optimization...\n")
res, final_params = optimize_params(
    df_fit=df_fit,
    r_script_path =r_script,
    param_names=param_names,
    initial=initial,
    bounds=bounds,
    scale=scale,
    workers=workers,
    log_file=log_file,
    maxiter=maxiter,
)

print("\n=== Optimization Complete ===")
print("Final parameters:")
for k, v in final_params.items():
    print(f"  {k}: {v}")

merged_final, _ = evaluate_model_parallel(df_fit, r_script, final_params, workers)

export_netcdf(merged_final, output_dir / "results.nc")
plot_rmse_bias(log_file, output_dir / "rmse_bias_plot.png")

print("\nDone. All outputs saved in:", output_dir)

In [ ]:
if __name__ == "__main__":
    main()


Loading dataset...
Splitting...

Starting optimization...



Δ-SNOW blocks:   0%|          | 0/123 [00:00<?, ?it/s]Process SpawnPoolWorker-4:
Process SpawnPoolWorker-3:
Process SpawnPoolWorker-2:
Process SpawnPoolWorker-1:
Process SpawnPoolWorker-5:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/Users/jakobwerkgarner/miniforge3/envs/MT_dsnow/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/jakobwerkgarner/miniforge3/envs/MT_dsnow/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/jakobwerkgarner/miniforge3/envs/MT_dsnow/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/jakobwerkgarner/miniforge3/envs/MT_dsnow/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/jakobwerkgarner/m

KeyboardInterrupt: 